# Joins

Most real analyses combine data from more than one place: transactions with customer records, ratings with movie titles, clicks with campaigns. A join does that. This notebook covers the join types Spark supports, the problems you are likely to run into, and how Spark actually moves data around to perform a join.

## Join expressions

A join brings together two DataFrames, the *left* and the *right*, by comparing the values of one or more *keys*. The most common join expression is an equi-join: it checks whether the key in the left row equals the key in the right row, and if so, puts the two rows together. Spark also supports more complicated join expressions, which we will see later.

## Join types

The join expression decides whether two rows match. The join type decides what ends up in the result:

* Inner joins keep only the rows whose keys exist in both DataFrames
* Outer (full) joins keep rows whose keys exist in either DataFrame
* Left outer joins keep every row of the left DataFrame
* Right outer joins keep every row of the right DataFrame
* Left semi joins keep the left rows that have a match on the right, but no columns from the right
* Left anti joins keep the left rows that have **no** match on the right
* Cross (Cartesian) joins match every left row with every right row

SQL also has natural joins, which join on all columns with the same name. They are easy to get wrong, and the DataFrame API has no method for them, so we skip them.

We will use three small DataFrames: people, the graduate programs they went to, and their status in the Spark project:

In [1]:
person = spark.createDataFrame([
    (0, "Bill Chambers", 0, [100]),
    (1, "Matei Zaharia", 1, [500, 250, 100]),
    (2, "Michael Armbrust", 1, [250, 100])])\
  .toDF("id", "name", "graduate_program", "spark_status")

graduateProgram = spark.createDataFrame([
    (0, "Masters", "School of Information", "UC Berkeley"),
    (2, "Masters", "EECS", "UC Berkeley"),
    (1, "Ph.D.", "EECS", "UC Berkeley")])\
  .toDF("id", "degree", "department", "school")

sparkStatus = spark.createDataFrame([
    (500, "Vice President"),
    (250, "PMC Member"),
    (100, "Contributor")])\
  .toDF("id", "status")

person.show()
graduateProgram.show()
sparkStatus.show()

+---+----------------+----------------+---------------+
| id|            name|graduate_program|   spark_status|
+---+----------------+----------------+---------------+
|  0|   Bill Chambers|               0|          [100]|
|  1|   Matei Zaharia|               1|[500, 250, 100]|
|  2|Michael Armbrust|               1|     [250, 100]|
+---+----------------+----------------+---------------+



+---+-------+--------------------+-----------+
| id| degree|          department|     school|
+---+-------+--------------------+-----------+
|  0|Masters|School of Informa...|UC Berkeley|
|  2|Masters|                EECS|UC Berkeley|
|  1|  Ph.D.|                EECS|UC Berkeley|
+---+-------+--------------------+-----------+



+---+--------------+
| id|        status|
+---+--------------+
|500|Vice President|
|250|    PMC Member|
|100|   Contributor|
+---+--------------+



Next, let's register these as temporary views so we can query them with SQL:

In [2]:
person.createOrReplaceTempView("person")
graduateProgram.createOrReplaceTempView("graduateProgram")
sparkStatus.createOrReplaceTempView("sparkStatus")

## Inner joins

An inner join keeps only the rows where the join expression is true. Here we match each person's `graduate_program` with the `id` of the program. We store the expression in a variable so we can reuse it:

In [3]:
joinExpression = person["graduate_program"] == graduateProgram["id"]

Inner is the default join type, so we only need the right DataFrame and the expression:

In [4]:
person.join(graduateProgram, joinExpression).show()

+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
| id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|  2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+



Nobody went to program 2 (the EECS master's), so it does not appear. We can also name the join type explicitly with the `how` argument:

In [5]:
person.join(graduateProgram,
            on = person["graduate_program"] == graduateProgram["id"],
            how = "inner").show()

+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
| id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|  2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+



The general format is:

```python
leftDF.join(rightDF,
            on = leftDF["abc"] == rightDF["xyz"],
            how = "joinType")
```

In SQL

```sql
SELECT * FROM person
INNER JOIN graduateProgram
ON person.graduate_program = graduateProgram.id
```

## Outer joins

A full outer join keeps every row from both DataFrames. Where a row has no match on the other side, Spark fills those columns with `null`:

In [6]:
person.join(graduateProgram,
            on = person["graduate_program"] == graduateProgram["id"],
            how = "outer").show()

+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
|   0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|   1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|   2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|NULL|            NULL|            NULL|           NULL|  2|Masters|                EECS|UC Berkeley|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+



`"outer"`, `"full"` and `"full_outer"` all mean the same thing.

In SQL
```sql
SELECT * FROM person
FULL OUTER JOIN graduateProgram
ON person.graduate_program = graduateProgram.id
```

## Left outer joins

A left outer join keeps every row of the left DataFrame and adds the matching columns from the right one, or `null` when there is no match. Here `graduateProgram` is on the left, so program 2 stays in even though no one is enrolled:

In [7]:
graduateProgram.join(person,
                     on = person["graduate_program"] == graduateProgram["id"],
                     how = "left_outer").show()

+---+-------+--------------------+-----------+----+----------------+----------------+---------------+
| id| degree|          department|     school|  id|            name|graduate_program|   spark_status|
+---+-------+--------------------+-----------+----+----------------+----------------+---------------+
|  0|Masters|School of Informa...|UC Berkeley|   0|   Bill Chambers|               0|          [100]|
|  1|  Ph.D.|                EECS|UC Berkeley|   2|Michael Armbrust|               1|     [250, 100]|
|  1|  Ph.D.|                EECS|UC Berkeley|   1|   Matei Zaharia|               1|[500, 250, 100]|
|  2|Masters|                EECS|UC Berkeley|NULL|            NULL|            NULL|           NULL|
+---+-------+--------------------+-----------+----+----------------+----------------+---------------+



In SQL
```sql
SELECT * FROM graduateProgram
LEFT OUTER JOIN person
ON person.graduate_program = graduateProgram.id
```

## Right outer joins

A right outer join is a left outer join with the two DataFrames swapped. The rows are the same, but the columns come out in a different order:

In [8]:
person.join(graduateProgram,
            on = person["graduate_program"] == graduateProgram["id"],
            how = "right_outer").show()

+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
|   0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|   2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|   1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|NULL|            NULL|            NULL|           NULL|  2|Masters|                EECS|UC Berkeley|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+



`"left"` and `"right"` are accepted as short names for `"left_outer"` and `"right_outer"`.

In SQL
```sql
SELECT * FROM person
RIGHT OUTER JOIN graduateProgram
ON person.graduate_program = graduateProgram.id
```

## Left semi joins

A semi join works like a filter. It returns the left rows that have at least one match on the right, and it never adds columns from the right DataFrame. Each left row appears at most once, even if it matches many right rows. Which programs have at least one student?

In [9]:
graduateProgram.join(person,
                     on = person["graduate_program"] == graduateProgram["id"],
                     how = "left_semi").show()

+---+-------+--------------------+-----------+
| id| degree|          department|     school|
+---+-------+--------------------+-----------+
|  0|Masters|School of Informa...|UC Berkeley|
|  1|  Ph.D.|                EECS|UC Berkeley|
+---+-------+--------------------+-----------+



Program 1 has two students, but it shows up once. An inner join would have returned it twice.

In SQL
```sql
SELECT * FROM graduateProgram
LEFT SEMI JOIN person
ON graduateProgram.id = person.graduate_program
```

## Left anti joins

An anti join is the opposite: it keeps the left rows that have **no** match on the right. It is the usual way to answer "which X never had a Y?" questions, such as customers who never ordered or products that never sold. Which programs have no students?

In [10]:
graduateProgram.join(person,
                     on = person["graduate_program"] == graduateProgram["id"],
                     how = "left_anti").show()

+---+-------+----------+-----------+
| id| degree|department|     school|
+---+-------+----------+-----------+
|  2|Masters|      EECS|UC Berkeley|
+---+-------+----------+-----------+



In SQL
```sql
SELECT * FROM graduateProgram
LEFT ANTI JOIN person
ON graduateProgram.id = person.graduate_program
```

## Cross (Cartesian) joins

A cross join has no join condition: it pairs every left row with every right row. Joining 3 people with 3 programs gives 9 rows. Joining two tables of 100,000 rows each gives 10 billion rows, so use cross joins only when you really mean it.

In [11]:
person.crossJoin(graduateProgram).show()

+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
| id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|  0|   Bill Chambers|               0|          [100]|  2|Masters|                EECS|UC Berkeley|
|  0|   Bill Chambers|               0|          [100]|  1|  Ph.D.|                EECS|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|  0|Masters|School of Informa...|UC Berkeley|
|  2|Michael Armbrust|               1|     [250, 100]|  0|Masters|School of Informa...|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|  2|Masters|                EECS|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC 

In SQL
```sql
SELECT * FROM person
CROSS JOIN graduateProgram
```

## Joins on complex types

A join expression can be any expression that returns true or false, not only an equality. Each person has an array of Spark status ids in `spark_status`. To look up the status names, we join where the array *contains* the status id:

In [12]:
from pyspark.sql.functions import expr

person.withColumnRenamed("id", "personId")\
  .join(sparkStatus, expr("array_contains(spark_status, id)"))\
  .select("personId", "name", "id", "status")\
  .show()

+--------+----------------+---+--------------+
|personId|            name| id|        status|
+--------+----------------+---+--------------+
|       0|   Bill Chambers|100|   Contributor|
|       1|   Matei Zaharia|500|Vice President|
|       1|   Matei Zaharia|250|    PMC Member|
|       1|   Matei Zaharia|100|   Contributor|
|       2|Michael Armbrust|250|    PMC Member|
|       2|Michael Armbrust|100|   Contributor|
+--------+----------------+---+--------------+



Matei has three ids in his array, so he matches three statuses and appears three times.

In SQL
```sql
SELECT person.id AS personId, name, sparkStatus.id, status
FROM person
INNER JOIN sparkStatus
ON array_contains(spark_status, sparkStatus.id)
```

# Challenges when using joins

## Handling duplicate column names

Duplicate column names in the result are one of the most common join headaches. Spark keeps track of each column with a unique internal id, but when you refer to a column by name, two columns with the same name are ambiguous.

This happens in two situations:

* The join keys have the same name in both DataFrames, and the join expression keeps both of them
* Two columns you are *not* joining on have the same name

Let's make a version of `graduateProgram` whose key has the same name as the key in `person`:

In [13]:
gradProgramDupe = graduateProgram.withColumnRenamed("id", "graduate_program")

df = person.join(gradProgramDupe,
                 on = person["graduate_program"] == gradProgramDupe["graduate_program"],
                 how = "inner")
df.show()

+---+----------------+----------------+---------------+----------------+-------+--------------------+-----------+
| id|            name|graduate_program|   spark_status|graduate_program| degree|          department|     school|
+---+----------------+----------------+---------------+----------------+-------+--------------------+-----------+
|  0|   Bill Chambers|               0|          [100]|               0|Masters|School of Informa...|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|               1|  Ph.D.|                EECS|UC Berkeley|
|  2|Michael Armbrust|               1|     [250, 100]|               1|  Ph.D.|                EECS|UC Berkeley|
+---+----------------+----------------+---------------+----------------+-------+--------------------+-----------+



There are now two `graduate_program` columns. Selecting one of them by name fails, because Spark cannot tell which one we mean:

In [14]:
try:
    df.select("graduate_program").show()
except Exception as e:
    print(type(e).__name__)
    print(str(e).split("\n")[0])

AnalysisException
[AMBIGUOUS_REFERENCE] Reference `graduate_program` is ambiguous, could be: [`graduate_program`, `graduate_program`].


There are three ways to fix this.

### Approach 1: a different join expression

When both keys have the same name, the easiest fix is to pass the column name (or a list of names) instead of a Boolean expression. Spark then keeps only one copy of the key:

In [15]:
person.join(gradProgramDupe, "graduate_program").show()

+----------------+---+----------------+---------------+-------+--------------------+-----------+
|graduate_program| id|            name|   spark_status| degree|          department|     school|
+----------------+---+----------------+---------------+-------+--------------------+-----------+
|               0|  0|   Bill Chambers|          [100]|Masters|School of Informa...|UC Berkeley|
|               1|  1|   Matei Zaharia|[500, 250, 100]|  Ph.D.|                EECS|UC Berkeley|
|               1|  2|Michael Armbrust|     [250, 100]|  Ph.D.|                EECS|UC Berkeley|
+----------------+---+----------------+---------------+-------+--------------------+-----------+



### Approach 2: drop the column after the join

You can also drop one of the duplicates after the join. Refer to it through the DataFrame it came from, so Spark knows which one to drop:

In [16]:
person.join(gradProgramDupe,
            person["graduate_program"] == gradProgramDupe["graduate_program"])\
  .drop(person["graduate_program"]).show()

+---+----------------+---------------+----------------+-------+--------------------+-----------+
| id|            name|   spark_status|graduate_program| degree|          department|     school|
+---+----------------+---------------+----------------+-------+--------------------+-----------+
|  0|   Bill Chambers|          [100]|               0|Masters|School of Informa...|UC Berkeley|
|  1|   Matei Zaharia|[500, 250, 100]|               1|  Ph.D.|                EECS|UC Berkeley|
|  2|Michael Armbrust|     [250, 100]|               1|  Ph.D.|                EECS|UC Berkeley|
+---+----------------+---------------+----------------+-------+--------------------+-----------+



### Approach 3: rename a column before the join

Or avoid the problem altogether by renaming one of the keys before the join:

In [17]:
gradProgram3 = graduateProgram.withColumnRenamed("id", "grad_id")

person.join(gradProgram3,
            person["graduate_program"] == gradProgram3["grad_id"]).show()

+---+----------------+----------------+---------------+-------+-------+--------------------+-----------+
| id|            name|graduate_program|   spark_status|grad_id| degree|          department|     school|
+---+----------------+----------------+---------------+-------+-------+--------------------+-----------+
|  0|   Bill Chambers|               0|          [100]|      0|Masters|School of Informa...|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|      1|  Ph.D.|                EECS|UC Berkeley|
|  2|Michael Armbrust|               1|     [250, 100]|      1|  Ph.D.|                EECS|UC Berkeley|
+---+----------------+----------------+---------------+-------+-------+--------------------+-----------+



## Duplicate keys multiply rows

If a key appears more than once on **both** sides, every left copy is matched with every right copy. A join can then return more rows than either input, and any sum you compute afterwards counts the same values several times.

Suppose a second table lists two advisors per program:

In [18]:
advisors = spark.createDataFrame([
    (1, "Alice"), (1, "Bob"),
    (0, "Carol"), (0, "Dan")])\
  .toDF("graduate_program", "advisor")

joined = person.join(advisors, "graduate_program")
joined.show()

print("person rows:", person.count(), "| advisors rows:", advisors.count(), "| joined rows:", joined.count())

+----------------+---+----------------+---------------+-------+
|graduate_program| id|            name|   spark_status|advisor|
+----------------+---+----------------+---------------+-------+
|               0|  0|   Bill Chambers|          [100]|  Carol|
|               0|  0|   Bill Chambers|          [100]|    Dan|
|               1|  1|   Matei Zaharia|[500, 250, 100]|  Alice|
|               1|  1|   Matei Zaharia|[500, 250, 100]|    Bob|
|               1|  2|Michael Armbrust|     [250, 100]|  Alice|
|               1|  2|Michael Armbrust|     [250, 100]|    Bob|
+----------------+---+----------------+---------------+-------+



person rows: 3 | advisors rows: 4 | joined rows: 6


Program 1 has two people and two advisors, so it produces 2 x 2 = 4 rows. Before a join, check whether your key is unique on each side, for example with `df.groupBy("key").count().where("count > 1")`. If it is not, aggregate first, then join.

## Null keys never match

`null == null` is not true in SQL (it is `null`), so rows with a null key are dropped by an inner join. When you do want nulls to match each other, use the null-safe equality `eqNullSafe` (written `<=>` in SQL):

In [19]:
left = spark.createDataFrame([(1, "a"), (None, "b")], "key int, left_val string")
right = spark.createDataFrame([(1, "x"), (None, "y")], "key int, right_val string")

print("with ==")
left.join(right, left["key"] == right["key"]).show()

print("with eqNullSafe")
left.join(right, left["key"].eqNullSafe(right["key"])).show()

with ==


+---+--------+---+---------+
|key|left_val|key|right_val|
+---+--------+---+---------+
|  1|       a|  1|        x|
+---+--------+---+---------+

with eqNullSafe


+----+--------+----+---------+
| key|left_val| key|right_val|
+----+--------+----+---------+
|NULL|       b|NULL|        y|
|   1|       a|   1|        x|
+----+--------+----+---------+



# How Spark performs joins

To understand how Spark performs a join, you need to know about two things: how the nodes talk to each other (the communication strategy), and how each node computes its part of the result. These details may not matter for every business problem, but knowing them helps when a join is slow or a job runs out of memory.

## Communication strategies

Spark mostly joins data in one of two ways. A **shuffle join** (in practice usually a *sort-merge join*) moves data so that rows with the same key end up on the same node, which means every node talks to every other node. A **broadcast join** sends a full copy of a small table to every node, so the big table never has to move. Spark also has a *shuffle hash join*, which it uses less often.

### Big table to big table

When both tables are big, Spark uses a sort-merge join. It has been the default strategy for large joins since Spark 2.3.

The illustration below shows how the data moves between executors to make the join possible:

<img src="https://github.com/soltaniehha/Big-Data-Analytics-for-Business/blob/master/figs/06-03-Joining-two-big-tables.png?raw=true" width="700" align="center"/>

In a shuffle join, every node sends each row to the node responsible for that row's key. That traffic is what makes these joins expensive, especially when the data is not spread evenly across keys.

Picture a company that receives billions of messages a day from Internet of Things devices and needs to compare each day's data with the previous day's, joining on device id, message type and date. Both DataFrames are big, so every worker node (and potentially every partition) has to communicate with the others during the join.

### Big table to small table

When one table is small enough to fit in the memory of a single worker node, with some room to spare, Spark can do better. It sends (broadcasts) a copy of the small table to every worker. After that, each worker joins its own part of the big table locally, without talking to any other node:

<img src="https://github.com/soltaniehha/Big-Data-Analytics-for-Business/blob/master/figs/06-03-A-broadcast-join.png?raw=true" width="700" align="center"/>

Broadcasting has a one-time cost at the start. After that there is no more communication between nodes, and every CPU works on its share of the join.

Spark broadcasts a table on its own when it estimates the table is smaller than `spark.sql.autoBroadcastJoinThreshold`. The Spark default is 10 MB, but Dataproc sets a higher value based on the size of your machines, so check yours:

In [20]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

'96m'

Spark does not know how big our hand-made DataFrames are before it runs the query, so the plan below starts out as a `SortMergeJoin`. You can see the shuffle step (`Exchange hashpartitioning`) on both sides of the join:

In [21]:
person.join(graduateProgram,
            person["graduate_program"] == graduateProgram["id"]).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [graduate_program#10L], [id#24L], Inner
   :- Sort [graduate_program#10L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(graduate_program#10L, 1000), ENSURE_REQUIREMENTS, [plan_id=2077]
   :     +- Project [_1#0L AS id#8L, _2#1 AS name#9, _3#2L AS graduate_program#10L, _4#3 AS spark_status#11]
   :        +- Filter isnotnull(_3#2L)
   :           +- Scan ExistingRDD[_1#0L,_2#1,_3#2L,_4#3]
   +- Sort [id#24L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(id#24L, 1000), ENSURE_REQUIREMENTS, [plan_id=2078]
         +- Project [_1#16L AS id#24L, _2#17 AS degree#25, _3#18 AS department#26, _4#19 AS school#27]
            +- Filter isnotnull(_1#16L)
               +- Scan ExistingRDD[_1#16L,_2#17,_3#18,_4#19]




`AdaptiveSparkPlan isFinalPlan=false` means Adaptive Query Execution (AQE) is on. AQE measures the real size of each side while the query runs and can switch to a broadcast join at that point.

If you already know a table is small, you can tell Spark to broadcast it with the `broadcast` hint around the small DataFrame. Now the plan uses a `BroadcastHashJoin` and the small side is a `BroadcastExchange`:

In [22]:
from pyspark.sql.functions import broadcast

person.join(broadcast(graduateProgram),
            person["graduate_program"] == graduateProgram["id"]).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [graduate_program#10L], [id#24L], Inner, BuildRight, false
   :- Project [_1#0L AS id#8L, _2#1 AS name#9, _3#2L AS graduate_program#10L, _4#3 AS spark_status#11]
   :  +- Filter isnotnull(_3#2L)
   :     +- Scan ExistingRDD[_1#0L,_2#1,_3#2L,_4#3]
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, true]),false), [plan_id=2110]
      +- Project [_1#16L AS id#24L, _2#17 AS degree#25, _3#18 AS department#26, _4#19 AS school#27]
         +- Filter isnotnull(_1#16L)
            +- Scan ExistingRDD[_1#16L,_2#17,_3#18,_4#19]




The same hint works in SQL:

In [23]:
spark.sql("""
SELECT /*+ BROADCAST(graduateProgram) */ *
FROM person
JOIN graduateProgram ON person.graduate_program = graduateProgram.id
""").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [graduate_program#10L], [id#24L], Inner, BuildRight, false
   :- Project [_1#0L AS id#8L, _2#1 AS name#9, _3#2L AS graduate_program#10L, _4#3 AS spark_status#11]
   :  +- Filter isnotnull(_3#2L)
   :     +- Scan ExistingRDD[_1#0L,_2#1,_3#2L,_4#3]
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, true]),false), [plan_id=2140]
      +- Project [_1#16L AS id#24L, _2#17 AS degree#25, _3#18 AS department#26, _4#19 AS school#27]
         +- Filter isnotnull(_1#16L)
            +- Scan ExistingRDD[_1#16L,_2#17,_3#18,_4#19]




Be careful: if you broadcast something that is too big, you can crash the driver, because the whole table is collected there before it is sent out.

### Little table to little table

When both tables are small, it is usually best to let Spark decide how to join them. You can still force a broadcast join if you notice strange behavior.